# COMP9098: Week 7 Lab, Grounding Oliver

**Real Tools, Real Data**

Format: individual, self-paced, nothing to write or edit. Just run each cell, read the output, and work through the reflection questions at the end.

**Companion files (upload these to the Colab file browser before running):** `staff_directory.csv`, `IT_Support_Handbook_Excerpt.pdf`


## Overview

Every tool Oliver has used so far, `deliver_dish`, `check_table_status`, `apply_discount`, is synthetic: a plain Python function returning a made-up value, with no real call leaving the notebook. That was deliberate, it let you focus on the agent loop, the Validator, and the Planner without also debugging a flaky API.

This week Oliver borrows three genuinely real tools, on loan from an unrelated internal-IT-helpdesk scenario, not his own restaurant, so nothing here is meant to resemble your own Project 1 or Project 2 agent. Watch what changes when a tool call leaves the notebook and touches something real:

1. **A real web lookup** -- Wikipedia's public API, actual internet, actual latency, actual chance of failure.
2. **A real structured file** -- `staff_directory.csv`, standing in for a small database.
3. **A real unstructured file** -- `IT_Support_Handbook_Excerpt.pdf`, standing in for a policy document or manual your agent might need to read.

All three are already wired up and working. There is nothing to complete. Run each cell top to bottom, look at what comes back, and think about whether your own project needs something in this shape.


## Setup

Run this once, you shouldn't need to edit it.

In [ ]:
# Eden AI, same setup as every previous week.
!pip install -q -U openai pypdf

import re, csv, json, time
from openai import OpenAI, __version__ as openai_version
print("openai library version:", openai_version)

import os
if not os.environ.get("EDEN_TOKEN"):
    from google.colab import userdata
    try:
        os.environ["EDEN_TOKEN"] = userdata.get("EDEN_TOKEN")
    except Exception as e:
        print("Could not read EDEN_TOKEN from Colab secrets:", e)

EDEN_TOKEN = os.environ["EDEN_TOKEN"].strip()
assert EDEN_TOKEN.startswith("sk-eden-"), "Check your EDEN_TOKEN value."

MODEL_ID = "qwen2.5-coder-32b-instruct"

client = OpenAI(api_key=EDEN_TOKEN, base_url="https://api.edenai.run/v3")
print("Client ready.")


## Tool 1: a real web lookup (Wikipedia)

`fetch_wikipedia_summary(topic)` calls Wikipedia's own public REST API and returns the real opening summary of a real article. No synthetic data anywhere in this function.

Notice three things that every synthetic tool so far has let you ignore:

- **Latency.** This takes noticeably longer than any local function call.
- **Failure is possible.** The topic might not exist, the request might time out, the API might be briefly unavailable. The function has to handle that instead of assuming success.
- **The content is untrusted.** Whatever comes back is real text written by someone else on the internet. It gets treated as data to read, never as instructions to follow, exactly like Week 6's Tools-surface discussion.

In [ ]:
import requests

def fetch_wikipedia_summary(topic: str, timeout: float = 6.0) -> dict:
    """Real web call: fetches the opening summary of a Wikipedia article.
    Returns a dict with ok/title/extract/url, or ok=False with an error reason.
    Never raises on a network problem, a well-behaved tool fails safe instead.
    """
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{requests.utils.quote(topic)}"
    try:
        resp = requests.get(url, timeout=timeout, headers={"User-Agent": "COMP9098-Lab/1.0"})
    except requests.exceptions.RequestException as e:
        return {"ok": False, "error": f"network error: {e}"}

    if resp.status_code == 404:
        return {"ok": False, "error": f"no Wikipedia article found for '{topic}'"}
    if resp.status_code != 200:
        return {"ok": False, "error": f"unexpected status {resp.status_code}"}

    data = resp.json()
    return {
        "ok": True,
        "title": data.get("title"),
        "extract": data.get("extract"),
        "url": data.get("content_urls", {}).get("desktop", {}).get("page"),
    }


In [ ]:
# Try a topic that exists, and one that doesn't. Look at both outcomes.
for topic in ["Prompt injection", "Munster Technological University", "zzzz_not_a_real_topic_zzzz"]:
    result = fetch_wikipedia_summary(topic)
    print("TOPIC:", topic)
    print(json.dumps(result, indent=2)[:500])
    print("-" * 60)


## Tool 2: a real structured file (CSV, standing in for a database)

`lookup_staff` reads `staff_directory.csv` from disk and filters it, the same shape of query you'd run against a real database table. Make sure you've uploaded `staff_directory.csv` to this Colab session's file browser (the folder icon on the left) before running this cell.

In [ ]:
def load_staff_directory(path: str = "staff_directory.csv") -> list:
    try:
        with open(path, newline="", encoding="utf-8") as f:
            return list(csv.DictReader(f))
    except FileNotFoundError:
        return []

STAFF = load_staff_directory()
print(f"Loaded {len(STAFF)} rows from staff_directory.csv" if STAFF
      else "staff_directory.csv not found, upload it to this Colab session first.")

def lookup_staff(department: str = None, role: str = None, name: str = None) -> list:
    """Real file lookup: filters the loaded CSV by any combination of department/role/name
    (case-insensitive substring match). Returns a list of matching rows (dicts)."""
    rows = STAFF
    if department:
        rows = [r for r in rows if department.lower() in r["department"].lower()]
    if role:
        rows = [r for r in rows if role.lower() in r["role"].lower()]
    if name:
        rows = [r for r in rows if name.lower() in r["name"].lower()]
    return rows


In [ ]:
for query in [
    {"department": "IT Support"},
    {"role": "Manager"},
    {"name": "Byrne"},
    {"department": "Marketing"},  # does not exist, watch what comes back
]:
    matches = lookup_staff(**query)
    print("QUERY:", query, "->", len(matches), "match(es)")
    for m in matches:
        print("   ", m["name"], "-", m["role"], "-", m["department"], "- ext.", m["extension"])
    print("-" * 60)


## Tool 3: a real unstructured file (PDF, standing in for a document your agent needs to read)

`search_handbook` loads `IT_Support_Handbook_Excerpt.pdf`, splits it into its six sections, and scores each section against your query by how many of the query's words it contains. This is a deliberately simple version of retrieval-augmented generation (RAG), a production system would usually rank with vector embeddings instead of word overlap, but the *pattern* is identical either way: pull the most relevant chunk out of a real document and hand only that chunk to the model, rather than the whole file. Upload `IT_Support_Handbook_Excerpt.pdf` to this Colab session before running this cell.

In [ ]:
from pypdf import PdfReader

def load_handbook_sections(path: str = "IT_Support_Handbook_Excerpt.pdf") -> list:
    try:
        reader = PdfReader(path)
    except FileNotFoundError:
        return []
    full_text = "\n".join(page.extract_text() or "" for page in reader.pages)
    # Split on the handbook's own "SECTION N:" headings.
    parts = re.split(r"(?=SECTION \d+:)", full_text)
    return [p.strip() for p in parts if p.strip().startswith("SECTION")]

HANDBOOK_SECTIONS = load_handbook_sections()
print(f"Loaded {len(HANDBOOK_SECTIONS)} sections from the handbook PDF." if HANDBOOK_SECTIONS
      else "IT_Support_Handbook_Excerpt.pdf not found, upload it to this Colab session first.")

STOPWORDS = {"the","a","an","is","are","to","of","in","on","for","and","or","i","my","how","do","does","what","if","it"}

def search_handbook(query: str, top_k: int = 1) -> list:
    """Real document search: scores each section by overlapping (non-stopword) words with the
    query, returns the top_k highest-scoring sections. Returns [] if nothing scores above 0."""
    query_words = {w for w in re.findall(r"[a-z']+", query.lower()) if w not in STOPWORDS}
    scored = []
    for section in HANDBOOK_SECTIONS:
        section_words = set(re.findall(r"[a-z']+", section.lower()))
        score = len(query_words & section_words)
        if score > 0:
            scored.append((score, section))
    scored.sort(key=lambda t: t[0], reverse=True)
    return [s for _, s in scored[:top_k]]


In [ ]:
for query in [
    "I lost my laptop, what do I do?",
    "how long does a new monitor take to arrive",
    "can I book a room in Block B",
]:
    hits = search_handbook(query, top_k=1)
    print("QUERY:", query)
    print(hits[0][:300] + "..." if hits else "(no matching section found)")
    print("-" * 60)


## Putting it together: one agent, three real tools, the same gate

This is the part worth watching closely. A planner decides which real tool (if any) a question needs, the chosen tool actually runs, real content comes back, and the same kind of `validate_action` gate you built in Week 4 still sits in front of execution, it doesn't care whether the tool underneath is synthetic or real.

In [ ]:
TOOL_SCHEMA = {
    "fetch_wikipedia_summary": 1,
    "lookup_staff": 1,   # a single JSON object argument with department/role/name keys
    "search_handbook": 1,
}

VALIDATION_LOG = []

def validate_action(tool_name: str, args, raised_by: str = "run_agent") -> tuple:
    """Same shape as Week 4's gate: checked once, before any tool runs."""
    entry = {"tool": tool_name, "args": args, "logged_at": time.time(), "raised_by": raised_by}
    if tool_name not in TOOL_SCHEMA:
        entry["reason"] = f"unknown tool: {tool_name}"
        VALIDATION_LOG.append(entry)
        return False, entry["reason"]
    entry["reason"] = "ok"
    VALIDATION_LOG.append(entry)
    return True, None

ROUTER_SYSTEM_PROMPT = """You route a user question to exactly one tool, or none.
Available tools:
- fetch_wikipedia_summary(topic): for general knowledge questions about the real world.
- lookup_staff(department=None, role=None, name=None): for questions about helpdesk/facilities/finance/HR staff.
- search_handbook(query): for questions about IT policy, procedure, or what to do in a situation.
Reply with exactly one line: TOOL: <tool_name> | ARGS: <a short JSON object>
If no tool fits, reply: TOOL: none"""

def call_router(user_message: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
    )
    return resp.choices[0].message.content.strip()

def run_agent(user_message: str) -> str:
    routing = call_router(user_message)
    print("ROUTER SAID:", routing)

    m = re.match(r"TOOL:\s*(\S+)\s*(?:\|\s*ARGS:\s*(.*))?", routing)
    if not m or m.group(1) == "none":
        return "(no tool needed for this question)"

    tool_name = m.group(1)
    try:
        args = json.loads(m.group(2)) if m.group(2) else {}
    except json.JSONDecodeError:
        args = {}

    ok, reason = validate_action(tool_name, args)
    if not ok:
        return f"BLOCKED by validate_action: {reason}"

    if tool_name == "fetch_wikipedia_summary":
        result = fetch_wikipedia_summary(args.get("topic", user_message))
        return result.get("extract", result.get("error", "no result"))
    elif tool_name == "lookup_staff":
        rows = lookup_staff(**args)
        return "\n".join(f"{r['name']} ({r['role']}, {r['department']})" for r in rows) or "no matches"
    elif tool_name == "search_handbook":
        hits = search_handbook(args.get("query", user_message))
        return hits[0] if hits else "no matching section"
    return "(unhandled tool)"


In [ ]:
for question in [
    "Who is the IT Support Manager?",
    "What should I do if I lose my work laptop?",
    "What is prompt injection?",
]:
    print("USER:", question)
    print("AGENT:", run_agent(question))
    print("=" * 70)

print()
print(f"VALIDATION_LOG now has {len(VALIDATION_LOG)} entries.")


## Part C: apply it to your own project

Nothing in this notebook is submitted or graded, and there is no new checkpoint attached to it, Project 1's four checkpoints are already done. This is purely to give you a working example of what a real tool looks like, in code, before Project 2.

If, after working through this, you think your own project genuinely needs one of these three shapes of tool, a real web lookup, a real structured file, or a real document search, or something else specific to your domain, email the lecturer with what you need and why. These three are the pre-approved default options; anything else needs a quick check first, the same way the external-API list already works for Project 2.

## Wrap-up reflection

Nothing here is submitted or graded. Answer for your own benefit, 3-5 sentences each.

1. Pick one of the three real tools above. What could actually go wrong with it in practice that could never happen with a synthetic tool, and did the code you just ran handle that, or just hope it wouldn't happen?
2. The Wikipedia summary, the staff directory rows, and the handbook section are all real content written by someone else. Where exactly in `run_agent` above does that content stop being "data the model reads" and risk becoming "instructions the model follows", and what currently stops that line from being crossed?
3. Looking at your own Project 1 or Project 2 agent idea, does it need a real tool in one of these three shapes? If yes, which one, and what is the smallest, most tightly-scoped version of it you could build, rather than the most powerful one?
